### 特徵工程 (Feature Engineering)

本檔案主要功能為建立特徵以供後續建模、定義 A/B 群組以供 SHAP 計算。

In [1]:
"""
Feature Engineering Pipeline for NBA Salary Valuation System
核心任務：將資料轉換為模型可用格式，並定義 Group A (實力) 與 Group B (市場雜訊) 分組。
"""

import pandas as pd
import numpy as np
import os
from typing import List, Tuple, Dict, Optional
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


def define_ab_groups(df: pd.DataFrame) -> Tuple[List[str], List[str]]:
    """
    動態分類 Group A 與 Group B。
    自動抓取所有帶有 _reg, _playoff 後綴的特徵。
    """
    # Group A: 純實力與狀態
    # 包含所有加權統計數據、年齡、傷病、季後賽經驗等
    group_a_features = ['age', 'has_playoff_exp']
    
    # 動態抓取所有統計特徵
    stat_cols = [col for col in df.columns if col.endswith('_reg') or col.endswith('_playoff')]
    group_a_features.extend(stat_cols)

    # Group B: 外部市場干擾
    # 包含 CBA 規則與市場供需變數
    group_b_features = [
        'is_retained',     # 是否與母隊續約 (鳥權溢價)
    ]

    return group_a_features, group_b_features


def create_interaction_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    基於合併後的特徵，建立有意義的互動特徵。
    """
    df = df.copy()

    # 1. 季後賽硬漢指數 (Playoff Elevation)
    # 若該球員有打季後賽，他的季後賽 VORP 或 BPM 是否高於例行賽？
    if 'VORP_reg' in df.columns and 'VORP_playoff' in df.columns:
        df['VORP_Elevation'] = np.where(
            df['has_playoff_exp'] == 1,
            df['VORP_playoff'] - df['VORP_reg'],
            0 # 沒打季後賽的，提升值為 0
        )
        logger.info("Created feature: VORP_Elevation")
        
    if 'BPM_reg' in df.columns and 'BPM_playoff' in df.columns:
         df['BPM_Elevation'] = np.where(
            df['has_playoff_exp'] == 1,
            df['BPM_playoff'] - df['BPM_reg'],
            0
        )
         logger.info("Created feature: BPM_Elevation")

    # 2. 季後賽時間佔比
    if 'MP_reg' in df.columns and 'MP_playoff' in df.columns:
        # 計算總上場時間 (粗估)
        total_mp_reg = df['MP_reg'] * df['G_reg']
        total_mp_playoff = df['MP_playoff'] * df['G_playoff']
        df['Playoff_MP_Ratio'] = np.where(
            total_mp_reg + total_mp_playoff > 0,
            total_mp_playoff / (total_mp_reg + total_mp_playoff),
            0
        )
        logger.info("Created feature: Playoff_MP_Ratio")

    return df

def prepare_features_for_modeling(
    df: pd.DataFrame,
    target_col: str = 'Cap_Pct'
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]: # 加上型別提示
    """
    為 XGBoost (預測定價) 與 Ordinal Logistic (預測年限) 準備最終的 X 和 y。
    """
    df = df.copy()
    
    # 確保不會有遺漏的 NA 值 (特別是互動特徵可能產生的)
    df = df.fillna(0)

    # 定義標識與目標欄位
    id_cols = ['Player', 'year']
    target_pct = target_col
    target_yrs = 'YRS'
    
    exclude_cols = id_cols + [target_pct, target_yrs]
    
    # 分離特徵矩陣、目標變數與識別欄位
    X = df.drop(columns=exclude_cols, errors='ignore')
    y_pct = df[target_pct] if target_pct in df.columns else None
    y_yrs = df[target_yrs] if target_yrs in df.columns else None
    
    # [修改處] 將識別欄位獨立為一個 DataFrame
    ids = df[id_cols] 

    logger.info(f"Prepared Feature Matrix (X) shape: {X.shape}")
    
    # 回傳 ids
    return X, y_pct, y_yrs, ids

def engineer_nba_features(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series, List[str], List[str]]:
    """
    一鍵執行特徵工程管線。
    輸入為 clean.py / merge 產出的合併 DataFrame。
    """
    logger.info("--- Starting Feature Engineering ---")

    # 1. 建立互動特徵
    df = create_interaction_features(df)
    
    # 2. 定義 A/B 組
    group_a, group_b = define_ab_groups(df)
    
    new_features = ['VORP_Elevation', 'BPM_Elevation', 'Playoff_MP_Ratio']
    group_a.extend([f for f in new_features if f in df.columns])
    
    # 3. 準備最終特徵矩陣與目標變數
    X, y_pct, y_yrs, ids = prepare_features_for_modeling(df) 
    
    # 確保 group 內的特徵都存在於 X 中
    group_a = [f for f in group_a if f in X.columns]
    group_b = [f for f in group_b if f in X.columns]
    
    logger.info(f"Final Group A (Skill) count: {len(group_a)}")
    logger.info(f"Final Group B (Noise) count: {len(group_b)}")
    logger.info("--- Feature Engineering Completed ---")
    
    return X, y_pct, y_yrs, ids, group_a, group_b

In [5]:
df = pd.read_csv('../../data/processed/merged_weighted_stats.csv')
X, y_pct, y_yrs, ids, group_a, group_b = engineer_nba_features(df)

print("準備匯出 API 專用的特徵資料庫...")

# 1. 將身分證 (ids) 與 特徵矩陣 (X) 水平合併
db_df = pd.concat([ids, X], axis=1)

# 2. 把目標變數也放進去 (雖然推論時用不到，但維持表格完整性)
if y_pct is not None:
    db_df['Cap_Pct'] = y_pct
if y_yrs is not None:
    db_df['YRS'] = y_yrs

# 3. 設定存檔路徑 (對齊你的專案結構)
# 確保指向 C:\Users\user\Python\Final_test\nba-salary-valuation\data\processed\
save_dir = r'C:\Users\user\Python\Final_test\nba-salary-valuation\data\processed'

# 如果資料夾不存在就建立一個
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

save_path = os.path.join(save_dir, 'featured_nba_data.csv')

# 4. 存檔！
db_df.to_csv(save_path, index=False)
print(f"✅ 特徵資料庫已成功儲存至: {save_path}")

2026-06-03 22:57:14,086 - INFO - --- Starting Feature Engineering ---


2026-06-03 22:57:14,096 - INFO - Created feature: VORP_Elevation
2026-06-03 22:57:14,098 - INFO - Created feature: BPM_Elevation
2026-06-03 22:57:14,100 - INFO - Created feature: Playoff_MP_Ratio
2026-06-03 22:57:14,110 - INFO - Prepared Feature Matrix (X) shape: (752, 96)
2026-06-03 22:57:14,111 - INFO - Final Group A (Skill) count: 95
2026-06-03 22:57:14,111 - INFO - Final Group B (Noise) count: 1
2026-06-03 22:57:14,111 - INFO - --- Feature Engineering Completed ---


準備匯出 API 專用的特徵資料庫...
✅ 特徵資料庫已成功儲存至: C:\Users\user\Python\Final_test\nba-salary-valuation\data\processed\featured_nba_data.csv
